# Caffeine & Sleep Time Planner (Set, Logic, Sequence)


In [1]:
import matplotlib.pyplot as plt

SLEEP_LIMIT = 50   # mg
DAILY_LIMIT = 400  # mg

def caffeine_left(dose, hours):
    return dose * 0.5 ** (hours / 5)

def show_time(hour):          # 25 -> "01:00"
    return f"{hour % 24:02d}:00"

In [2]:
def make_sets(dose, drink, bed, wake):
    if bed < drink:        # bedtime after midnight = next day
        bed += 24
    while wake <= bed:     # wake-up must come after bedtime
        wake += 24

    A = set()
    for h in range(drink, wake + 1):
        if caffeine_left(dose, h - drink) <= SLEEP_LIMIT:
            A.add(h)
    B = set(range(bed, wake))
    return A, B

In [3]:
def check_sleep(dose, A, B):
    p = B <= A                 # B ⊆ A
    q = dose <= DAILY_LIMIT
    return p, q

In [4]:
R = 0.5 ** (1 / 5)   # r ≈ 0.87

def max_dose(drink, bed):
    if bed < drink:          # bedtime after midnight
        bed += 24
    n = bed - drink
    limit = SLEEP_LIMIT / R ** n
    return min(int(limit + 1e-9), DAILY_LIMIT)

In [5]:
def ask_number(text, low, high, input_fn=input):
    while True:
        user_input = input_fn(text)

        if not user_input.isdigit():
            print(f"Please enter a whole number from {low} to {high}.")
            continue

        value = int(user_input)

        if low <= value <= high:
            return value
        else:
            print(f"Please enter a whole number from {low} to {high}.")

# accept times written with a dot or a colon
def ask_time(text, input_fn=input):
    while True:
        user_input = input_fn(text).strip()

        try:
            # accept both 21.00 and 21:00
            user_input = user_input.replace(".", ":")

            if ":" in user_input:
                parts = user_input.split(":")

                if len(parts) != 2:
                    raise ValueError

                hour = int(parts[0])
                minute = int(parts[1])

                # this project works in whole hours
                if minute != 0:
                    raise ValueError

            else:
                hour = int(user_input)

            if 0 <= hour <= 23:
                return hour

        except ValueError:
            pass

        print("Please enter an hour from 0 to 23, e.g. 21, 21.00 or 21:00.")

def plan(dose, drink, bed, wake, show_plot=True):
    A, B = make_sets(dose, drink, bed, wake)
    p, q = check_sleep(dose, A, B)

    # User-friendly output
    if not p:
        print("\u274c Caffeine is still above 50 mg at bedtime")

    if not q:
        print("\u274c Over the 400 mg daily limit")

    if p and q:
        print("\u2705 You can sleep at your planned time")

    if A:
        print("\u23f0 Earliest safe sleep time:", show_time(min(A)))

    # Sequence
    print(f"\u2615 To sleep at {show_time(bed)}, drink at most {max_dose(drink, bed)} mg (you drank {dose} mg)")

    # Graph
    if show_plot:
        hours = list(range(drink, max(B) + 2))
        plt.plot(hours, [caffeine_left(dose, h - drink) for h in hours], marker="o", label="Caffeine")
        plt.axhline(SLEEP_LIMIT, color="red", linestyle="--", label="50 mg limit")
        plt.axvline(min(B), color="gray", linestyle=":", label="Bedtime")
        plt.xticks(hours, [show_time(h) for h in hours], rotation=60)
        plt.ylabel("Caffeine (mg)")
        plt.legend()
        plt.grid(True)
        plt.show()

    return p, q

def main():
    dose = ask_number("Caffeine amount (mg): ", 1, 1000)
    drink = ask_time("Drink time (21:00): ")
    bed = ask_time("Planned bedtime (23:00): ")
    wake = ask_time("Wake-up time (07:00): ")

    plan(dose, drink, bed, wake)


## Test: 3 cases

In [ ]:
print("=== Case 1: 96 mg at 12:00, bed 22:00 ===")
assert plan(96, 12, 22, 6, show_plot=False) == (True, True)     # can sleep

print("\n=== Case 2: 96 mg at 20:00, bed 23:00 ===")
assert plan(96, 20, 23, 7, show_plot=False) == (False, True)    # not p

print("\n=== Case 3: 450 mg at 06:00, bed 23:00 ===")
assert plan(450, 6, 23, 7, show_plot=False) == (True, False)    # not q

# Sequence: maximum dose in mg
assert max_dose(12, 22) == 200
assert max_dose(20, 23) == 75
assert max_dose(6, 23) == 400   # capped by the 400 mg/day limit

# dose <= max_dose  <=>  p and q (matches the Logic part)
for dose, drink, bed, wake in [(96, 12, 22, 6), (96, 20, 23, 7), (450, 6, 23, 7)]:
    p, q = check_sleep(dose, *make_sets(dose, drink, bed, wake))
    assert (dose <= max_dose(drink, bed)) == (p and q)


=== Case 1: 96 mg at 12:00, bed 22:00 ===
✅ You can sleep at your planned time
⏰ Earliest safe sleep time: 17:00
☕ To sleep at 22:00, drink at most 200 mg (you drank 96 mg)

=== Case 2: 96 mg at 20:00, bed 23:00 ===
❌ Caffeine is still above 50 mg at bedtime
⏰ Earliest safe sleep time: 01:00
☕ To sleep at 23:00, drink at most 75 mg (you drank 96 mg)

=== Case 3: 450 mg at 06:00, bed 23:00 ===
❌ Over the 400 mg daily limit
⏰ Earliest safe sleep time: 22:00
☕ To sleep at 23:00, drink at most 400 mg (you drank 450 mg)


## Enter input :

In [ ]:
main()